Member 2's Task (Preprocessing & Feature Engineerin)

Step 1: Data Selection & Cleaning

Removing unnecessary columns and those that could lead to data leakage.

In [ ]:

# Data Selection & Cleaning
# Selecting necessary columns including pre-match bookmaker odds (B365H, B365D, B365A).
columns_to_keep = [
    'id', 'date', 'league_name', 'home_team_long_name', 'away_team_long_name',
    'home_team_goal', 'away_team_goal', 'target',
    'B365H', 'B365D', 'B365A'
]

available_cols = [col for col in columns_to_keep if col in df_matches.columns]
df_clean = df_matches[available_cols].copy()

df_clean.rename(columns={
    'home_team_long_name': 'home_team',
    'away_team_long_name': 'away_team'
}, inplace=True)

print("Step 1 Complete - Cleaned data shape:", df_clean.shape)
display(df_clean.head())

#convert date
df_matches['date'] = pd.to_datetime(
    df_matches['date'],
    errors='coerce'
)


print("\nDate range:")
print("Start:", df_matches['date'].min())
print("End:", df_matches['date'].max())

# Viewing the first 5 rows
print(df_clean.head())

#Data leakage check

post_match_variables = [
    col for col in df_matches.columns
    if any(keyword in col.lower() for keyword in [
        'goal',
        'shot',
        'corner',
        'foul',
        'card',
        'possession',
        'cross'
    ])
]

print("Potential post-match variables:")
for col in post_match_variables:
    print("-", col)

leakage_check = pd.DataFrame({
    'Variable': post_match_variables,
    'Potential Leakage': 'Yes',
    'Reason': 'Match event/statistic may only be known during or after the match'
})

display(leakage_check)

#Step 2: Feature Engineering (Rolling Form & Goal Difference)
# Sorting by date chronologically
df_clean['date'] = pd.to_datetime(df_clean['date'])
df_clean = df_clean.sort_values('date')

df_clean.head()

# Separating data into groups for team-level history tracking
home_df = df_clean[['date', 'home_team', 'home_team_goal', 'away_team_goal']].copy().rename(
    columns={'home_team': 'team', 'home_team_goal': 'goals_scored', 'away_team_goal': 'goals_conceded'}
)
away_df = df_clean[['date', 'away_team', 'away_team_goal', 'home_team_goal']].copy().rename(
    columns={'away_team': 'team', 'away_team_goal': 'goals_scored', 'home_team_goal': 'goals_conceded'}
)
home_df.head()

# Combining into a single team history log
team_history = pd.concat([home_df, away_df]).sort_values(['team', 'date'])
team_history.head()

# Rolling Form Calculation (Goals Scored, Goals Conceded, and Goal Difference)
team_history['rolling_goals_scored'] = team_history.groupby('team')['goals_scored'].transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
team_history['rolling_goals_conceded'] = team_history.groupby('team')['goals_conceded'].transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())

team_history.head()

# Added Goal Difference feature for stronger team strength representation
team_history['rolling_goal_diff'] = team_history['rolling_goals_scored'] - team_history['rolling_goals_conceded']

# Merging rolling features back to the main dataset for home team
df_features = pd.merge(
    df_clean,
    team_history[['date', 'team', 'rolling_goals_scored', 'rolling_goals_conceded', 'rolling_goal_diff']],
    left_on=['date', 'home_team'], right_on=['date', 'team'], how='left'
).rename(columns={
    'rolling_goals_scored': 'home_form_scored',
    'rolling_goals_conceded': 'home_form_conceded',
    'rolling_goal_diff': 'home_form_goal_diff'
}).drop('team', axis=1)

# Merging rolling features back to the main dataset for away team
df_features = pd.merge(
    df_features,
    team_history[['date', 'team', 'rolling_goals_scored', 'rolling_goals_conceded', 'rolling_goal_diff']],
    left_on=['date', 'away_team'], right_on=['date', 'team'], how='left'
).rename(columns={
    'rolling_goals_scored': 'away_form_scored',
    'rolling_goals_conceded': 'away_form_conceded',
    'rolling_goal_diff': 'away_form_goal_diff'
}).drop('team', axis=1)

# Ensure the 'id' column from df_matches is carried over
if 'id' not in df_features.columns and 'id' in df_clean.columns:
    df_features['id'] = df_clean['id']

# Cleaning initial matches where rolling metrics or odds might be missing
subset_cols = ['home_form_scored', 'away_form_scored']
if 'B365H' in df_features.columns:
    subset_cols.extend(['B365H', 'B365D', 'B365A'])

df_features.dropna(subset=subset_cols, inplace=True)
print("Step 2 Complete - Features added (including Goal Diff & Odds). New shape:", df_features.shape)
print("Columns in df_features:", df_features.columns.tolist())


#Step 3: Categorical Encoding
# Selecting feature columns and target variable
feature_cols = [
    'league_name', 'home_team', 'away_team',
    'home_form_scored', 'home_form_conceded', 'home_form_goal_diff',
    'away_form_scored', 'away_form_conceded', 'away_form_goal_diff'
]

if 'B365H' in df_features.columns:
    feature_cols.extend(['B365H', 'B365D', 'B365A'])

X = df_features[feature_cols].copy()
y = df_features['target']
# Viewing data selected for X
X.head()

from sklearn.preprocessing import LabelEncoder

# Encoding League names
le_league = LabelEncoder()
X['league_name'] = le_league.fit_transform(X['league_name'])

# Check if the league has switched to using numbers.
X[['league_name']].head()

# Encoding Team names
le_team = LabelEncoder()
all_teams = pd.concat([X['home_team'], X['away_team']])
le_team.fit(all_teams)

X['home_team'] = le_team.transform(X['home_team'])
X['away_team'] = le_team.transform(X['away_team'])

print("Step 3 Complete - Encoding done.")
display(X.head())

#Step 4: Train / Test Split

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

print("Step 4 Complete - Data split.")
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

#Step 5: Feature Scaling

# Selecting all numerical features to scale (rolling form, goal diff, and pre-match odds)
numerical_cols = [
    'home_form_scored', 'home_form_conceded', 'home_form_goal_diff',
    'away_form_scored', 'away_form_conceded', 'away_form_goal_diff'
]

if 'B365H' in X_train.columns:
    numerical_cols.extend(['B365H', 'B365D', 'B365A'])

scaler = StandardScaler()

# Fitting strictly to training data and transforming train set (preventing data leakage)
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])

# Transforming test data using training parameters only
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

print("Step 5 Complete - Scaling done. Final Training Data:")
display(X_train.head())




